# Binary Image Classification Using a Convolutional Neural Network

This beginner-friendly project classifies casting product images into two quality classes:

- `ok_front` -> Non-defective -> Class 0
- `def_front` -> Defective -> Class 1

The notebook works in Google Colab or Jupyter Notebook. Update `DATASET_SOURCE_DIR` in Section 2 to point to the Kaggle dataset when needed.

## Section 1: Import Libraries
We import TensorFlow, image utilities, plotting tools, and evaluation metrics.

In [ ]:
import os
import random
import shutil
from pathlib import Path

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('TensorFlow version:', tf.__version__)

## Section 2: Configuration
Images are resized to 224 x 224, a common size that keeps useful visual detail while remaining practical for a beginner project. A batch size of 32 balances speed and memory. The model may stop before 25 epochs when validation loss stops improving.

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 25
SEED = 42

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
MODELS_DIR = PROJECT_ROOT / 'models'
GRAPHS_DIR = PROJECT_ROOT / 'outputs' / 'graphs'
PREDICTIONS_DIR = PROJECT_ROOT / 'outputs' / 'predictions'
for directory in (DATA_DIR, MODELS_DIR, GRAPHS_DIR, PREDICTIONS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Update this path if the Kaggle archive was extracted somewhere else.
DATASET_SOURCE_DIR = PROJECT_ROOT / 'dataset'
CLASS_NAMES = ['ok_front', 'def_front']
CLASS_LABELS = {0: 'Non-defective', 1: 'Defective'}
print('Project root:', PROJECT_ROOT.resolve())
print('Dataset source:', DATASET_SOURCE_DIR.resolve())
print('Class mapping:', CLASS_NAMES[0], '-> 0,', CLASS_NAMES[1], '-> 1')

## Section 3: Dataset Loading
The Kaggle dataset may already have train, validation, and test folders. If it does not, the next code cell creates a reproducible 70/15/15 split from folders named `ok_front` and `def_front`. The `class_names` argument fixes the required mapping instead of relying on alphabetical order.

In [ ]:
# Remove only empty placeholder split folders so the splitter can initialize fresh data.
for split_name in ('train', 'validation', 'test'):
    split_dir = DATA_DIR / split_name
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}
    has_images = any(path.is_file() and path.suffix.lower() in image_extensions for path in split_dir.rglob('*')) if split_dir.exists() else False
    if split_dir.exists() and not has_images:
        shutil.rmtree(split_dir)

In [ ]:
def image_files(folder):
    extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}
    return sorted([path for path in folder.rglob('*') if path.is_file() and path.suffix.lower() in extensions])

def prepare_dataset_splits(source_dir, output_dir):
    split_names = ['train', 'validation', 'test']
    if all((output_dir / split_name).is_dir() for split_name in split_names):
        print('Using existing data/train, data/validation, and data/test folders.')
        return
    if not source_dir.exists():
        raise FileNotFoundError(
            f'Dataset not found at {source_dir}. Extract the Kaggle dataset there or update DATASET_SOURCE_DIR.'
        )
    source_classes = [source_dir / class_name for class_name in CLASS_NAMES]
    if not all(folder.is_dir() for folder in source_classes):
        raise FileNotFoundError('Expected source folders: ' + ', '.join(str(folder) for folder in source_classes))
    rng = np.random.default_rng(SEED)
    for class_name, class_dir in zip(CLASS_NAMES, source_classes):
        files = image_files(class_dir)
        if not files:
            raise ValueError(f'No image files found in {class_dir}')
        shuffled = list(files)
        rng.shuffle(shuffled)
        train_end = int(0.70 * len(shuffled))
        validation_end = train_end + int(0.15 * len(shuffled))
        groups = {
            'train': shuffled[:train_end],
            'validation': shuffled[train_end:validation_end],
            'test': shuffled[validation_end:],
        }
        for split_name, split_files in groups.items():
            destination = output_dir / split_name / class_name
            destination.mkdir(parents=True, exist_ok=True)
            for source_file in split_files:
                destination_file = destination / source_file.name
                if not destination_file.exists():
                    shutil.copy2(source_file, destination_file)
    print('Created reproducible 70% train, 15% validation, 15% test splits.')

prepare_dataset_splits(DATASET_SOURCE_DIR, DATA_DIR)

train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'train', class_names=CLASS_NAMES, label_mode='binary',
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED
)
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'validation', class_names=CLASS_NAMES, label_mode='binary',
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=False
)
test_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'test', class_names=CLASS_NAMES, label_mode='binary',
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=False
)
print('Class names:', train_dataset.class_names)
print('Training batches:', tf.data.experimental.cardinality(train_dataset).numpy())
print('Validation batches:', tf.data.experimental.cardinality(validation_dataset).numpy())
print('Test batches:', tf.data.experimental.cardinality(test_dataset).numpy())
print('Dataset element specification:', train_dataset.element_spec)

## Section 4: Visualize Dataset Images
Viewing examples helps confirm that the folders and labels are correct before training.

In [ ]:
plt.figure(figsize=(12, 12))
for images, labels in train_dataset.take(1):
    for index in range(min(9, len(images))):
        axis = plt.subplot(3, 3, index + 1)
        plt.imshow(images[index].numpy().astype('uint8'))
        label = int(labels[index].numpy()[0])
        plt.title(f'{CLASS_LABELS[label]} (class {label})')
        plt.axis('off')
plt.tight_layout()
plt.show()

## Section 5: Performance Optimization
Prefetching prepares the next batch while the model trains on the current batch, reducing waiting time in the input pipeline.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

## Section 6: Data Augmentation
Augmentation changes training images slightly, which helps the model learn useful defect patterns instead of memorizing exact images. It is placed inside the model, so it is active only during training.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10)
], name='data_augmentation')

## Section 7: Image Normalization
Rescaling converts pixel values from approximately 0-255 to 0-1, which makes optimization more stable.

In [ ]:
normalization = tf.keras.layers.Rescaling(1.0 / 255)

## Section 8: Build the CNN Model
The convolution layers learn increasingly complex visual features. Global average pooling keeps the classifier smaller than a large Flatten layer.

In [ ]:
def build_cnn(dropout_rate=0.40):
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(*IMAGE_SIZE, 3)),
        data_augmentation,
        normalization,
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(dropout_rate),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
        {
            "cell_type": "markdown",
            "metadata": {"language": "markdown"},
            "source": ["## Section 28: Final Conclusion\n", "Run the next cell after training to generate a human-readable conclusion from actual metrics. It does not invent numerical results."]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "metadata": {"language": "python"},
            "outputs": [],
            "source": [
                "print(f\"The CNN learned to classify the casting images with test accuracy {test_results['accuracy']:.2%}, precision {test_results['precision']:.2%}, and recall {test_results['recall']:.2%}.\")\n",
                "print(f\"Defective-product recall was {defective_recall:.2%}; this matters because false negatives can allow defective products to pass inspection.\")\n",
                "print(f\"The training-history check found overfitting: {overfitting_observed}. Data augmentation and dropout were used to improve generalization.\")\n",
                "print('A possible future improvement is transfer learning with a pretrained model and a larger, more diverse inspection dataset.')"
            ]
        }
The parameter counts show how many values the model learns during training.

In [ ]:
model.summary()
print('Total parameters:', model.count_params())
print('Trainable parameters:', sum(np.prod(weight.shape) for weight in model.trainable_weights))
print('Non-trainable parameters:', sum(np.prod(weight.shape) for weight in model.non_trainable_weights))

## Section 10: Explain Architecture Design Choices
1. **CNN:** It learns edges, textures, shapes, and defect patterns directly from images.
2. **224 x 224:** This preserves useful detail while controlling computation.
3. **32 -> 64 -> 128 filters:** Deeper layers learn more varied and complex features.
4. **3 x 3 kernels:** Small local windows are efficient and capture nearby patterns.
5. **ReLU:** It is simple, fast, and helps neural networks learn non-linear patterns.
6. **MaxPooling:** It reduces feature-map size and keeps strong responses.
7. **GlobalAveragePooling2D:** It reduces parameters and overfitting compared with Flatten.
8. **Dropout 0.40:** It randomly disables 40% of selected activations during training to regularize the model.
9. **Sigmoid:** It produces one probability for the defective class in a binary problem.

## Section 11: Compile the Model
Adam adapts updates for each parameter. A learning rate of 0.001 is a useful starting point. Binary cross-entropy matches two-class prediction. Accuracy measures all correct decisions, while precision measures how many predicted defects are truly defective and recall measures how many actual defects were found.

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

## Section 12: Callbacks
Early stopping restores the best validation model. Reducing the learning rate can help when progress slows. ModelCheckpoint saves the best model to disk.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2
    ),
    tf.keras.callbacks.ModelCheckpoint(
        MODELS_DIR / 'best_model.keras', monitor='val_loss',
        save_best_only=True
    )
]

## Section 13: Train the Model
Training can finish before 25 epochs because EarlyStopping watches validation loss.

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=25,
    callbacks=callbacks
)
completed_epochs = len(history.history['loss'])
print(f'Training complete: {completed_epochs} epoch(s) completed.')

## Section 14: Plot Accuracy Graph

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'accuracy.png', dpi=150)
plt.show()

## Section 15: Plot Loss Graph

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'loss.png', dpi=150)
plt.show()

## Section 16: Analyze Training Results
The observations below are calculated from the actual history. A gap is reported rather than judged from hardcoded values.

In [ ]:
training_accuracy = history.history['accuracy']
validation_accuracy = history.history['val_accuracy']
final_train_accuracy = training_accuracy[-1]
final_validation_accuracy = validation_accuracy[-1]
accuracy_gap = abs(final_train_accuracy - final_validation_accuracy)
best_validation_epoch = int(np.argmax(validation_accuracy)) + 1
early_stopping_activated = completed_epochs < EPOCHS
overfitting_observed = accuracy_gap > 0.10 and final_train_accuracy > final_validation_accuracy
print('Training accuracy improved:', training_accuracy[-1] > training_accuracy[0])
print('Validation accuracy improved:', validation_accuracy[-1] > validation_accuracy[0])
print(f'Final accuracy gap: {accuracy_gap:.3f}')
print('Large training/validation gap:', accuracy_gap > 0.10)
print('Evidence of overfitting:', overfitting_observed)
print('EarlyStopping activated:', early_stopping_activated)
print('Epochs completed:', completed_epochs)
print('Best validation accuracy epoch:', best_validation_epoch)

## Section 17: Evaluate the Model

In [ ]:
test_results = model.evaluate(test_dataset, return_dict=True, verbose=1)
print(f"Test Loss: {test_results['loss']:.4f}")
print(f"Test Accuracy: {test_results['accuracy']:.4f}")
print(f"Test Precision: {test_results['precision']:.4f}")
print(f"Test Recall: {test_results['recall']:.4f}")

## Section 18: Generate Predictions
The test dataset is not shuffled. Therefore, labels collected in iteration order match the order returned by `model.predict(test_dataset)`.

In [ ]:
actual_labels = np.concatenate([labels.numpy().ravel() for _, labels in test_dataset]).astype(int)
probabilities = model.predict(test_dataset)
predictions = (probabilities.flatten() >= 0.5).astype(int)
print('Actual labels:', len(actual_labels))
print('Predictions:', len(predictions))
assert len(actual_labels) == len(predictions), 'Labels and predictions are not aligned.'

## Section 19: Confusion Matrix
A True Negative is a correctly identified non-defective product. A False Positive is an approved product incorrectly sent for inspection. A False Negative is a defective product incorrectly approved. A True Positive is a correctly detected defective product. False Negatives are especially important because a defective product could pass quality control.

In [ ]:
matrix = confusion_matrix(actual_labels, predictions, labels=[0, 1])
plt.figure(figsize=(6, 5))
plt.imshow(matrix, interpolation='nearest', cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ['Non-defective', 'Defective'])
plt.yticks(tick_marks, ['Non-defective', 'Defective'])
plt.xlabel('Predicted label')
plt.ylabel('Actual label')
for row in range(2):
    for column in range(2):
        plt.text(column, row, matrix[row, column], ha='center', va='center')
plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'confusion_matrix.png', dpi=150)
plt.show()
true_negative, false_positive, false_negative, true_positive = matrix.ravel()
print(f'TN={true_negative}, FP={false_positive}, FN={false_negative}, TP={true_positive}')

## Section 20: Classification Report

In [ ]:
print(classification_report(
    actual_labels, predictions, labels=[0, 1],
    target_names=['Non-defective', 'Defective'], zero_division=0
))

## Section 21: Recall for Defective Products
A False Negative occurs when a defective product is predicted as non-defective. This matters because the product may pass inspection. Recall for the defective class is important because it measures the proportion of all actual defective products that the system successfully detects. In quality control, missing a defect can be more costly than sending a good product for manual inspection.

In [ ]:
report = classification_report(
    actual_labels, predictions, labels=[0, 1],
    target_names=['Non-defective', 'Defective'], output_dict=True, zero_division=0
)
defective_recall = report['Defective']['recall']
print(f'Defective recall: {defective_recall:.4f}')

## Section 22: Test Five Unseen Images
This function applies the same resize and probability threshold used by the test evaluation. It chooses five test images as a convenient unseen-image demonstration; replace `unseen_image_paths` with external images when available.

In [ ]:
def predict_single_image(image_path, trained_model=model):
    image = Image.open(image_path).convert('RGB').resize(IMAGE_SIZE)
    image_array = np.asarray(image, dtype=np.float32)
    batch = np.expand_dims(image_array, axis=0)
    probability = float(trained_model.predict(batch, verbose=0)[0][0])
    is_defective = probability >= 0.5
    prediction = 'Defective' if is_defective else 'Non-defective'
    action = 'Send for manual inspection' if is_defective else 'Approved'
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.title(f'Prediction: {prediction}')
    plt.axis('off')
    plt.show()
    print(f'Image: {image_path.name}')
    print(f'Prediction: {prediction}')
    print(f'Probability: {probability * 100:.2f}%')
    print(f'Action: {action}')
    return prediction, probability

unseen_image_paths = image_files(DATA_DIR / 'test')[:5]
if len(unseen_image_paths) < 5:
    print('At least five test images are required for this demonstration.')
else:
    for unseen_path in unseen_image_paths:
        predict_single_image(unseen_path)

## Section 23: Save the Final Model

In [ ]:
final_model_path = MODELS_DIR / 'final_cnn_model.keras'
model.save(final_model_path)
loaded_model = tf.keras.models.load_model(final_model_path)
loaded_test_results = loaded_model.evaluate(test_dataset, return_dict=True, verbose=0)
print('Saved model:', final_model_path.resolve())
print(f"Loaded model test accuracy: {loaded_test_results['accuracy']:.4f}")

## Section 24: Design Decision Table
| Design Decision | Selected Value | Reason |
|---|---|---|
| Image size | 224 x 224 | Balance between detail and computation |
| Problem type | Binary classification | Two output classes |
| Model type | CNN | Suitable for image data |
| Conv filters | 32, 64, 128 | Learn increasingly complex features |
| Kernel size | 3 x 3 | Efficient local feature extraction |
| Hidden activation | ReLU | Efficient and commonly used |
| Pooling | MaxPooling | Reduces feature dimensions |
| Output activation | Sigmoid | Produces binary probability |
| Optimizer | Adam | Adaptive and beginner-friendly |
| Learning rate | 0.001 | Good starting value |
| Loss | Binary Cross-Entropy | Suitable for binary classification |
| Batch size | 32 | Balance between speed and memory |
| Epochs | Maximum 25 | Enough training with early stopping |
| Dropout | 0.40 | Helps reduce overfitting |
| Augmentation | Flip, rotation, zoom, contrast | Improves robustness |
| Metrics | Accuracy, Precision, Recall | Overall and defect detection performance |

## Section 25: Bonus Experiment
The experiment changes only dropout from 0.40 to 0.20 and uses the same datasets and training settings. Results are read from actual evaluation output.

In [ ]:
bonus_model = build_cnn(dropout_rate=0.20)
bonus_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])
bonus_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]
bonus_history = bonus_model.fit(train_dataset, validation_data=validation_dataset, epochs=EPOCHS, callbacks=bonus_callbacks)
bonus_results = bonus_model.evaluate(test_dataset, return_dict=True, verbose=0)
comparison = {
    'Metric': ['Accuracy', 'Precision', 'Recall'],
    'Original dropout 0.40': [test_results['accuracy'], test_results['precision'], test_results['recall']],
    'Changed dropout 0.20': [bonus_results['accuracy'], bonus_results['precision'], bonus_results['recall']]
}
try:
    import pandas as pd
    comparison_table = pd.DataFrame(comparison)
    display(comparison_table)
except ImportError:
    print(comparison)
print('Original Design: Dropout = 0.40')
print('Changed Design: Dropout = 0.20')
print('Reason for Change: Test whether weaker regularization improves or reduces generalization.')
print(f"Original Accuracy: {test_results['accuracy']:.4f}")
print(f"New Accuracy: {bonus_results['accuracy']:.4f}")
print('Observation: Compare the displayed values; the result depends on the dataset and training run.')

## Section 26: README.md
A project README is included beside this notebook. It contains setup instructions, architecture details, evaluation guidance, and placeholders for values that are available only after training.

## Section 27: requirements.txt
The project requirements file lists TensorFlow, NumPy, Matplotlib, scikit-learn, Pillow, and Pandas for the optional comparison table.

## Section 28: Final Conclusion
Run this final cell after training to generate a human-readable conclusion from actual metrics. It does not invent numerical results.

The CNN learned to classify the casting images with a test accuracy of **{test_results['accuracy']:.2%}**, precision of **{test_results['precision']:.2%}**, and recall of **{test_results['recall']:.2%}**. Defective-product recall is especially important because false negatives can allow defective products to pass inspection. Based on the training-history check, overfitting was **{overfitting_observed}**. Data augmentation and dropout were used to improve generalization. A possible future improvement is transfer learning with a pretrained model and a larger, more diverse inspection dataset.